In [1]:
import pandas as pd
import numpy as np

# ---- File paths (edit if your files are named differently / in a different folder) ----
PHASE2_FILE = "Phase2_Consensus_Ranking_Calculations.xlsx"
PHASE3_FILE = "phase3_results.xlsx"
OUTPUT_FILE = "phase5_results.xlsx"

# ---- 1. Load Phase 2 & Phase 3 outputs ----
p2_consensus = pd.read_excel(PHASE2_FILE, sheet_name="Phase2_Consensus")

p3_summary  = pd.read_excel(PHASE3_FILE, sheet_name="Phase3_Summary")       # Average_Spearman, Average_Kendall per method
p3_topk     = pd.read_excel(PHASE3_FILE, sheet_name="Top_K_Agreement")      # Top10/Top20 agreement % per method
p3_spearman = pd.read_excel(PHASE3_FILE, sheet_name="Spearman", index_col=0)  # 14x14 correlation matrix

methods = list(p3_spearman.columns)
print(f"{len(methods)} methods found:", methods)

# ---- 2. Compute Phase 5 metrics (Steps 21-23) ----
results = []

for m in methods:
    # Step 21: Average Correlation Score
    avg_spearman = p3_summary.loc[p3_summary["Method"] == m, "Average_Spearman"].values[0]
    avg_kendall  = p3_summary.loc[p3_summary["Method"] == m, "Average_Kendall"].values[0]
    avg_corr_score = (avg_spearman + avg_kendall) / 2

    # Step 22: Consistency Score
    # std() uses ddof=1 (sample std), matching Excel's STDEV().
    # Includes the method's self-correlation (=1.0), a constant offset shared by
    # every method's row, so it doesn't bias the relative comparison.
    consistency = 1 - p3_spearman.loc[m].std()

    # Step 23: Stability Score
    # RMSE between this method's own ranks and Overall_Consensus_Rank (Phase 2),
    # across every phone, normalized by the number of phones.
    rmse = np.sqrt(np.mean((p2_consensus[m] - p2_consensus["Overall_Consensus_Rank"]) ** 2))
    stability = 1 - (rmse / len(p2_consensus))

    top10 = p3_topk.loc[p3_topk["Method"] == m, "Top10_Agreement_%"].values[0]
    top20 = p3_topk.loc[p3_topk["Method"] == m, "Top20_Agreement_%"].values[0]

    # Overall Performance Score — simple average of all sub-scores, each rescaled to 0-1
    overall_score = np.mean([avg_corr_score, consistency, stability, top10 / 100, top20 / 100])

    results.append({
        "Method": m,
        "Average_Spearman": avg_spearman,
        "Average_Kendall": avg_kendall,
        "Average_Correlation_Score": avg_corr_score,
        "Consistency_Score": consistency,
        "Stability_Score": stability,
        "Top10_Agreement_%": top10,
        "Top20_Agreement_%": top20,
        "Overall_Performance_Score": overall_score,
    })

phase5_df = pd.DataFrame(results)
phase5_df["Overall_Performance_Rank"] = (
    phase5_df["Overall_Performance_Score"].rank(ascending=False, method="min").astype(int)
)
phase5_df = phase5_df.sort_values("Overall_Performance_Rank").reset_index(drop=True)

# ---- 3. Export ----
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    phase5_df.to_excel(writer, sheet_name="Phase5_Method_Performance", index=False)

print(f"Saved {OUTPUT_FILE}")
phase5_df

14 methods found: ['MARCOS', 'SECA', 'MUTRISS', 'MAIRCA', 'CODAS', 'RAFSI', 'CRADIS', 'CoCoSo', 'MABAC', 'COPRAS', 'RAM', 'WASPAS', 'PIV', 'WISP']
Saved phase5_results.xlsx


,Method,Average_Spearman,Average_Kendall,Average_Correlation_Score,Consistency_Score,Stability_Score,Top10_Agreement_%,Top20_Agreement_%,Overall_Performance_Score,Overall_Performance_Rank
0,PIV,0.902088,0.804335,0.853211,0.918989,0.919179,100,100.0,0.938276,1
1,SECA,0.925824,0.838913,0.882369,0.909166,0.959960,100,90.0,0.930299,2
2,MAIRCA,0.925824,0.838913,0.882369,0.909166,0.959960,100,90.0,0.930299,2
3,CoCoSo,0.897708,0.766515,0.832111,0.923821,0.913826,100,85.0,0.903952,4
4,WASPAS,0.937941,0.852093,0.895017,0.915804,0.976370,80,90.0,0.897438,5
5,RAM,0.937808,0.858551,0.898179,0.910413,0.976048,60,85.0,0.846928,6
6,MUTRISS,0.932565,0.842181,0.887373,0.907582,0.961882,50,85.0,0.821367,7
7,MARCOS,0.912020,0.823987,0.868003,0.887817,0.948551,50,80.0,0.800874,8
8,CRADIS,0.912020,0.823987,0.868003,0.887817,0.948551,50,80.0,0.800874,8
9,MABAC,0.912020,0.823987,0.868003,0.887817,0.948551,50,80.0,0.800874,8
